<a href="https://colab.research.google.com/github/jashwanth-cse/Jashwanth-Codeboosters-Internship-2026/blob/main/Phase_01_Data_Engineering/Day_03_ETL_Pandas_APIs/Day_03_ETL_Pandas_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests --quiet
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
raw_df=pd.read_csv('/content/drive/MyDrive/messy_sales_data.csv')
print(f"Dataset loaded:{raw_df.shape[0]}students,{raw_df.shape[1]} columns")
print(raw_df.columns.tolist())

Dataset loaded:30students,9 columns
['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


In [ ]:
print("Data Quality Diagnosis Report")
print("-------------------------------")
print("No of missing values in each column:")
print(raw_df.isnull().sum())
print("Duplicate Rows:",raw_df.duplicated().sum())
print("-------------------------------")
print("Data Types:")
print(raw_df.dtypes)

print("Unique Categories:",raw_df['category'].unique())
print("Sample customer name",raw_df['customer_name'].dropna().unique()[:8])
print("Sample order date:",raw_df['order_date'].unique()[:6])

Data Quality Diagnosis Report
-------------------------------
No of missing values in each column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
Duplicate Rows: 0
-------------------------------
Data Types:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object
Unique Categories: ['Electronics' 'Accessories' nan]
Sample customer name ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
Sample order date: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [ ]:
copy_df=raw_df.copy()
print("Working copy created")

print("raw_df is untouched- we can always reset by running df=raw_df.copy()")

Working copy created
raw_df is untouched- we can always reset by running df=raw_df.copy()


Median -Data's center value(middle value)


In [ ]:
# Fix #1: Handle Missing Values

print("Before fixing nulls:",raw_df.isnull().sum().sum(),'total missing values')


#Fix missing quantity -fill with null customer names

raw_df['customer_name'].fillna('Unknown customer',inplace=True)

median_qty=raw_df['quantity'].median()
raw_df['quantity'].fillna(median_qty,inplace=True)
print("Filled missing quantity with median",median_qty)

print("After fixing nulls:",raw_df.isnull().sum().sum(),'total missing values')

Before fixing nulls: 7 total missing values
Filled missing quantity with median 2.0
After fixing nulls: 2 total missing values


In [ ]:
print("Before duplication",len(raw_df),"rows")
#Show duplicate rows
print("Duplicate rows ",raw_df.duplicated().sum())
print(raw_df[raw_df.duplicated(keep=False)][['order_id','customer_name','product','order_date']])

#Remove exact duplicate rows
raw_df.drop_duplicates(inplace=True)
print("After deduplication",len(raw_df),"rows")
print("Rows removed",len(copy_df)-len(raw_df))

Before duplication 30 rows
Duplicate rows  0
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []
After deduplication 30 rows
Rows removed 0


In [ ]:
print("Sample dates before parsing")
print(raw_df['order_date'].head(8).tolist())
raw_df['order_date']=pd.to_datetime(
    raw_df['order_date'],
    dayfirst=False,
    errors='coerce' #if parsing fails put NaT
)

raw_df['year']=raw_df['order_date'].dt.year
raw_df['month']=raw_df['order_date'].dt.month
raw_df['month_name']=raw_df['order_date'].dt.strftime("%B")

print("\nSample dates after parsing")
print(raw_df[['order_date','year','month','month_name']].head(10))

Sample dates before parsing
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

Sample dates after parsing
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January
5        NaT     NaN    NaN        NaN
6 2024-01-12  2024.0    1.0    January
7 2024-01-13  2024.0    1.0    January
8 2024-01-15  2024.0    1.0    January
9 2024-01-15  2024.0    1.0    January


In [ ]:
#Fix 4 Standardization
print("Before standardization",raw_df['customer_name'].unique()[:6])

raw_df['customer_name']=(
    raw_df['customer_name']
    .str.strip()
    .str.title()
)

print("Before keyboard rows with Electronics category ")
wrong_mask=(raw_df['product']=='keyboard')&(raw_df['category']=="Electronics")

print(raw_df[wrong_mask],[['product','category']])

raw_df.loc[wrong_mask,'category']='Accessories'

Before standardization ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
Before keyboard rows with Electronics category 
Empty DataFrame
Columns: [order_id, customer_name, product, category, quantity, unit_price, order_date, city, sales_rep, year, month, month_name]
Index: [] [['product', 'category']]


In [ ]:
raw_df['quantity']=pd.to_numeric(raw_df['quantity'],errors='coerce')
raw_df['unit_price']=pd.to_numeric(raw_df['unit_price'],errors='coerce')
raw_df['revenue']=raw_df['quantity']*raw_df['unit_price']
print('Revenue column created:')
print(raw_df[['customer_name','product','quantity','unit_price','revenue']].head())
print(f"\n Total Revenue across all orders: rs{raw_df['revenue'].sum():,.0f}")

Revenue column created:
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop       2.0       45000  90000.0
1    Priya Nair       NaN       1.0       15000  15000.0
2    Amit Verma  Keyboard       3.0        1200   3600.0
3  Sunita Patel   Monitor       2.0       22000  44000.0
4  Ramesh Kumar    Laptop       2.0       45000  90000.0

 Total Revenue across all orders: rs818,000


In [ ]:
print("POST CLEANING VALIDATION REPORT")
print("Original rows:",len(copy_df))
print("Cleaned rows:",len(raw_df))
print("Rows removed:",len(raw_df)-len(copy_df))
print("Missing values",raw_df.isnull().sum().sum())
print("Duplicates:",raw_df.duplicated().sum())
print("Date nulls",raw_df['order_date'].isnull().sum())
print("Revenue NaN",raw_df['revenue'].isnull().sum())
print("Categories",sorted(raw_df['category'].unique()))

all_clean=(
    raw_df.isnull().sum().sum()==0 and
    raw_df.duplicated().sum()==0
)

POST CLEANING VALIDATION REPORT
Original rows: 30
Cleaned rows: 30
Rows removed: 0
Missing values 10
Duplicates: 0
Date nulls 2
Revenue NaN 0


TypeError: '<' not supported between instances of 'float' and 'str'

In [ ]:
product_rev=(
    raw_df.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
)

print("Revenue by Product")

In [ ]:
API_KEY='a601765a8cc2881c45eeabdc5a9696a0'
BASE_URL='https://api.openweathermap.org/data/2.5/weather'
CITIES=["Mumbai","Hyderabad","Chennai","Rajapalayam"]


1. What are the three stages of ETL? Describe each stage using an example from today's sales dataset.
2. A DataFrame has 500 rows. After calling df.dropna(), it has 412 rows. What does this tell you?
3. Write code to remove duplicates from df where 'same row' means same customer_name AND same product.
4. What is the difference between fillna(9) and fillna(df['col'].median()) ? When would you prefer each?
5. Write Python code to call the weather API for 'Delhi' and print the temperature in Celsius.
Q6: What does response.status_code == 200 mean? What should you do when the code is 401?